# GraphSAGE Fault Propagation
This notebook demonstrates the injection of synthetic faults and training a heterogeneous GNN to predict downstream risk scores.

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.optim as optim
from backend.app.data.factory_graph import build_synthetic_factory_graph, convert_to_pyg_heterodata
from backend.app.data.fault_injector import inject_synthetic_faults
from backend.app.data.graph_schema import get_edge_types, NODE_MACHINE
from backend.app.models.gnn_layer import HeteroEquipmentGNN, FaultPredictionHead
from backend.app.training.gnn_trainer import GNNFaultTrainer

## 1. Setup Graph and Inject Faults

In [ ]:
# Generate graph with 3 assembly lines and 4 machines per line
nx_graph = build_synthetic_factory_graph(num_lines=3, machines_per_line=4)
machine_embeddings = torch.randn((12, 256))
data = convert_to_pyg_heterodata(nx_graph, machine_embeddings)

# Inject a synthetic fault starting at machine 0
data = inject_synthetic_faults(data, root_fault_node=0, root_node_type=NODE_MACHINE, max_hops=3, p_fault_base=0.9)
print("Machine Fault Labels:", data[NODE_MACHINE].y)

## 2. Train GraphSAGE

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on device: {device}")

gnn = HeteroEquipmentGNN(hidden_channels=128, out_channels=64, edge_types=get_edge_types()).to(device)
head = FaultPredictionHead(in_channels=64).to(device)
optimizer = optim.Adam(list(gnn.parameters()) + list(head.parameters()), lr=0.01)

data = data.to(device)
trainer = GNNFaultTrainer(gnn, head, optimizer)

# Mock training loop
for epoch in range(50):
    metrics = trainer.train_step(data, target_node_type=NODE_MACHINE)
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Loss {metrics['loss']:.4f}, Accuracy {metrics['accuracy']:.4f}")

## 3. Evaluation (Hop-Distance Accuracy)

In [ ]:
eval_metrics = trainer.evaluate(data, target_node_type=NODE_MACHINE)
print(f"Final Accuracy: {eval_metrics['accuracy']:.4f}")
print("Predictions:", eval_metrics['predictions'])
print("Ground Truth:", data[NODE_MACHINE].y)
